# 📊 Notebook 5: Portfolio Optimization
**FinTech Stock Market Analysis — Optimization Techniques**

This notebook bridges **Data Analytics** and **Optimization Techniques** by applying classical calculus-based optimization to portfolio construction:

| Topic | Application |
|---|---|
| **Critical Points** | Find minimum-variance allocation analytically (2-asset case) |
| **Applied Optimization** | Maximize Sharpe ratio / minimize portfolio variance |
| **Multivariable Optimization** | n-asset Markowitz optimization with `scipy.optimize` + Lagrange multipliers |

### The Markowitz Problem
Given $n$ assets with daily returns, find weights $\mathbf{w} = [w_1, \ldots, w_n]$ that:
$$\min_{\mathbf{w}} \; \mathbf{w}^\top \Sigma \mathbf{w} \quad \text{subject to} \quad \sum_{i=1}^n w_i = 1, \; w_i \geq 0$$
or equivalently, maximize the **Sharpe ratio**: $\max_{\mathbf{w}} \dfrac{\mu_p - r_f}{\sigma_p}$

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

RF_RATE = 0.05  # 5% annual risk-free rate
COLORS  = ['#00e5ff','#00e676','#ff6b6b','#ffd740','#ea80fc','#69f0ae','#ff5252','#82b1ff']

# Load all closing prices
closes = pd.read_csv('../data/all_closes.csv', index_col=0, parse_dates=True)
closes.dropna(inplace=True)   # keep only rows where ALL stocks have data

daily_returns = closes.pct_change().dropna()
mean_returns  = daily_returns.mean()
cov_matrix    = daily_returns.cov()
tickers       = list(closes.columns)
n_assets      = len(tickers)

print(f'✅ Loaded {len(closes)} trading days × {n_assets} stocks')
print(f'   Period : {closes.index[0].date()} → {closes.index[-1].date()}')
print(f'   Stocks : {tickers}')
print(f'\nAnnualized mean returns:')
for t, r in (mean_returns * 252 * 100).items():
    print(f'  {t:6s}: {r:+.2f}%')

## Part 1 — Critical Points: 2-Asset Minimum Variance

For a portfolio of **two assets** with weight $w$ in Asset 1 and $(1-w)$ in Asset 2:

$$\sigma_p^2(w) = w^2\sigma_1^2 + (1-w)^2\sigma_2^2 + 2w(1-w)\sigma_{12}$$

**Critical point** — set the first derivative to zero:
$$\frac{d\sigma_p^2}{dw} = 2w\sigma_1^2 - 2(1-w)\sigma_2^2 + 2(1-2w)\sigma_{12} = 0$$

Solving for $w$:
$$\boxed{w^* = \frac{\sigma_2^2 - \sigma_{12}}{\sigma_1^2 + \sigma_2^2 - 2\sigma_{12}}}$$

The **second derivative** $\frac{d^2\sigma_p^2}{dw^2} = 2\sigma_1^2 + 2\sigma_2^2 - 4\sigma_{12} > 0$ confirms it is a **minimum**.

In [ ]:
t1, t2 = 'JPM', 'V'
s1_sq  = daily_returns[t1].var()
s2_sq  = daily_returns[t2].var()
s12    = daily_returns[t1].cov(daily_returns[t2])

w_star         = (s2_sq - s12) / (s1_sq + s2_sq - 2 * s12)
w_star_clipped = float(np.clip(w_star, 0, 1))
second_deriv   = 2*s1_sq + 2*s2_sq - 4*s12

print('📐 CRITICAL POINTS — 2-Asset Minimum Variance')
print('='*55)
print(f'  σ²({t1}) = {s1_sq:.8f}')
print(f'  σ²({t2}) = {s2_sq:.8f}')
print(f'  σ₁₂     = {s12:.8f}')
print(f'\n  w* = ({s2_sq:.6f} − {s12:.6f}) / ({s1_sq:.6f} + {s2_sq:.6f} − {2*s12:.6f})')
print(f'  w* = {w_star:.6f}  (clipped to feasible range: {w_star_clipped:.4f})')
print(f'\n  2nd derivative = {second_deriv:.8f} > 0  ✅ Confirmed minimum')
print(f'\n  Optimal: {w_star_clipped*100:.1f}% in {t1},  {(1-w_star_clipped)*100:.1f}% in {t2}')

# Plot σ²_p(w) and mark the critical point
w_range = np.linspace(0, 1, 400)
var_curve = (w_range**2 * s1_sq
             + (1 - w_range)**2 * s2_sq
             + 2 * w_range * (1 - w_range) * s12) * 252
var_min_pt = (w_star_clipped**2*s1_sq
              + (1-w_star_clipped)**2*s2_sq
              + 2*w_star_clipped*(1-w_star_clipped)*s12) * 252

fig_cp = go.Figure()
fig_cp.add_trace(go.Scatter(
    x=w_range*100, y=var_curve,
    mode='lines', line=dict(color='#00e5ff', width=2.5),
    name='σ²_p(w)'
))
fig_cp.add_vline(
    x=w_star_clipped*100, line_dash='dash', line_color='#ffd740', opacity=0.85,
    annotation_text=f"w* = {w_star_clipped:.3f}",
    annotation_font_color='#ffd740'
)
fig_cp.add_trace(go.Scatter(
    x=[w_star_clipped*100], y=[var_min_pt],
    mode='markers', marker=dict(size=14, color='#ff6b6b', symbol='star'),
    name=f'Critical Point (Minimum)'
))
fig_cp.update_layout(
    template='plotly_dark', height=400,
    title=f'Critical Points: {t1}/{t2} Portfolio Variance vs Weight w₁',
    xaxis_title=f'Allocation in {t1} (%)',
    yaxis_title='Annualized Portfolio Variance'
)
fig_cp.show()

## Part 2 — Multivariable Optimization: Lagrange Multipliers (n Assets)

For $n$ assets, the minimum variance problem becomes:

$$\min_{\mathbf{w}} \; \mathbf{w}^\top \Sigma \mathbf{w} \quad \text{s.t.} \quad \mathbf{1}^\top \mathbf{w} = 1$$

**Lagrangian:**
$$\mathcal{L}(\mathbf{w}, \lambda) = \mathbf{w}^\top \Sigma \mathbf{w} - \lambda\left(\mathbf{1}^\top \mathbf{w} - 1\right)$$

**KKT (first-order) condition:** $\dfrac{\partial \mathcal{L}}{\partial \mathbf{w}} = 2\Sigma\mathbf{w} - \lambda\mathbf{1} = \mathbf{0}$

Solving: $\mathbf{w}^* = \dfrac{\lambda}{2}\Sigma^{-1}\mathbf{1}$, where $\lambda = \dfrac{2}{\mathbf{1}^\top \Sigma^{-1} \mathbf{1}}$

In [ ]:
Sigma    = cov_matrix.values
ones_vec = np.ones(n_assets)

try:
    Sigma_inv   = np.linalg.inv(Sigma)
    lambda_val  = 2.0 / (ones_vec @ Sigma_inv @ ones_vec)
    w_lagrange  = (lambda_val / 2.0) * (Sigma_inv @ ones_vec)
    # Apply non-negativity: project to simplex
    w_lagrange  = np.clip(w_lagrange, 0, None)
    w_lagrange /= w_lagrange.sum()

    r_lag = np.dot(w_lagrange, mean_returns) * 252
    v_lag = np.sqrt(w_lagrange @ (cov_matrix.values * 252) @ w_lagrange)

    print('📊 LAGRANGE MULTIPLIER — Analytical Minimum Variance')
    print('='*55)
    print(f'  λ = 2 / (1ᵀ Σ⁻¹ 1) = {lambda_val:.8f}')
    print(f'  w* = (λ/2) · Σ⁻¹ · 1  (then projected to w_i ≥ 0)')
    print(f'\nOptimal Weights:')
    for t, w in zip(tickers, w_lagrange):
        bar = '█' * int(w * 40)
        print(f'  {t:6s}: {w*100:5.1f}%  {bar}')
    print(f'\nPortfolio Performance:')
    print(f'  Annualized Return     : {r_lag*100:+.2f}%')
    print(f'  Annualized Volatility : {v_lag*100:.2f}%')
    print(f'  Sharpe Ratio          : {(r_lag - RF_RATE)/v_lag:.4f}')
except np.linalg.LinAlgError:
    print('⚠️  Singular covariance matrix — proceeding with numerical optimization.')

## Part 3 — Monte Carlo Simulation of Random Portfolios

Before running the optimizer, we visualize the **feasible set** by sampling 10,000 random weight vectors.

In [ ]:
def portfolio_performance(weights):
    """Return annualized (return, volatility, Sharpe) for a weight vector."""
    r = np.dot(weights, mean_returns) * 252
    v = np.sqrt(weights @ (cov_matrix.values * 252) @ weights)
    s = (r - RF_RATE) / v if v > 1e-10 else 0.0
    return r, v, s

N_SIM = 10_000
np.random.seed(42)

mc_r = np.zeros(N_SIM)
mc_v = np.zeros(N_SIM)
mc_s = np.zeros(N_SIM)
mc_w = np.zeros((N_SIM, n_assets))

for i in range(N_SIM):
    w = np.random.random(n_assets)
    w /= w.sum()
    mc_w[i]  = w
    mc_r[i], mc_v[i], mc_s[i] = portfolio_performance(w)

best_mc_idx = mc_s.argmax()
print(f'✅ Simulated {N_SIM:,} random portfolios')
print(f'\nBest random portfolio (Sharpe = {mc_s[best_mc_idx]:.3f}):')
print(f'  Return     : {mc_r[best_mc_idx]*100:+.2f}%')
print(f'  Volatility : {mc_v[best_mc_idx]*100:.2f}%')
print(f'  Weights    : { {t: f"{w*100:.1f}%" for t, w in zip(tickers, mc_w[best_mc_idx])} }')

## Part 4 — Applied Optimization with `scipy.optimize.minimize`

We use the **SLSQP** (Sequential Least Squares Programming) method — a gradient-based solver for constrained nonlinear optimization.

**Problem A — Maximize Sharpe Ratio** (equivalent to minimize negative Sharpe):
$$\min_{\mathbf{w}} -\frac{\mu_p - r_f}{\sigma_p} \quad \text{s.t.} \quad \sum w_i = 1, \; 0 \leq w_i \leq 1$$

**Problem B — Minimize Variance:**
$$\min_{\mathbf{w}} \; \sigma_p = \sqrt{\mathbf{w}^\top \Sigma \mathbf{w}} \quad \text{s.t.} \quad \sum w_i = 1, \; 0 \leq w_i \leq 1$$

In [ ]:
x0          = np.full(n_assets, 1.0 / n_assets)   # equal-weight starting point
bounds      = tuple((0.0, 1.0) for _ in range(n_assets))
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}

# ── Problem A: Maximum Sharpe Ratio ──────────────────────────────────────────
res_ms = minimize(
    fun=lambda w: -portfolio_performance(w)[2],
    x0=x0, method='SLSQP', bounds=bounds, constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)
w_ms = res_ms.x
ms_r, ms_v, ms_s = portfolio_performance(w_ms)

print('🏆 MAXIMUM SHARPE RATIO PORTFOLIO')
print('='*50)
print(f'  Converged  : {res_ms.success}  ({res_ms.nit} iterations)')
print(f'  Return     : {ms_r*100:+.2f}%')
print(f'  Volatility : {ms_v*100:.2f}%')
print(f'  Sharpe     : {ms_s:.4f}')
print(f'\n  Weights:')
for t, w in zip(tickers, w_ms):
    bar = '█' * int(w * 40)
    print(f'    {t:6s}: {w*100:5.1f}%  {bar}')

print()

# ── Problem B: Minimum Variance ───────────────────────────────────────────────
res_mv = minimize(
    fun=lambda w: portfolio_performance(w)[1],
    x0=x0, method='SLSQP', bounds=bounds, constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)
w_mv = res_mv.x
mv_r, mv_v, mv_s = portfolio_performance(w_mv)

print('🛡️  MINIMUM VARIANCE PORTFOLIO')
print('='*50)
print(f'  Converged  : {res_mv.success}  ({res_mv.nit} iterations)')
print(f'  Return     : {mv_r*100:+.2f}%')
print(f'  Volatility : {mv_v*100:.2f}%')
print(f'  Sharpe     : {mv_s:.4f}')
print(f'\n  Weights:')
for t, w in zip(tickers, w_mv):
    bar = '█' * int(w * 40)
    print(f'    {t:6s}: {w*100:5.1f}%  {bar}')

## Part 5 — Efficient Frontier

The **Efficient Frontier** is the set of portfolios that achieve the minimum possible variance for each level of target return — a parametric curve solved by adding one extra equality constraint:

$$\min_{\mathbf{w}} \; \sigma_p \quad \text{s.t.} \quad \sum w_i = 1, \; \mu_p = \bar{\mu}, \; w_i \geq 0$$

We sweep $\bar{\mu}$ from the minimum to maximum feasible return.

In [ ]:
target_returns = np.linspace(mc_r.min(), mc_r.max(), 80)
ef_vols, ef_rets = [], []

for target in target_returns:
    cons_ef = [
        {'type': 'eq', 'fun': lambda w:        np.sum(w) - 1.0},
        {'type': 'eq', 'fun': lambda w, t=target: portfolio_performance(w)[0] - t}
    ]
    res = minimize(
        fun=lambda w: portfolio_performance(w)[1],
        x0=x0, method='SLSQP', bounds=bounds, constraints=cons_ef,
        options={'maxiter': 500, 'ftol': 1e-10}
    )
    if res.success:
        r_ef, v_ef, _ = portfolio_performance(res.x)
        ef_rets.append(r_ef * 100)
        ef_vols.append(v_ef * 100)

print(f'✅ Efficient frontier: {len(ef_vols)} optimal portfolios computed')

In [ ]:
# ── Master Visualization ──────────────────────────────────────────────────────
fig_ef = go.Figure()

# 1. Monte Carlo cloud — coloured by Sharpe ratio
fig_ef.add_trace(go.Scatter(
    x=mc_v * 100, y=mc_r * 100,
    mode='markers',
    marker=dict(
        size=3, opacity=0.45,
        color=mc_s, colorscale='Viridis',
        colorbar=dict(title='Sharpe', x=1.02)
    ),
    name='Random Portfolios',
    hovertemplate='Vol: %{x:.1f}%<br>Return: %{y:.1f}%<extra></extra>'
))

# 2. Efficient frontier
if ef_vols:
    fig_ef.add_trace(go.Scatter(
        x=ef_vols, y=ef_rets,
        mode='lines', line=dict(color='white', width=3),
        name='Efficient Frontier'
    ))

# 3. Optimal portfolios
fig_ef.add_trace(go.Scatter(
    x=[ms_v * 100], y=[ms_r * 100],
    mode='markers+text',
    marker=dict(size=18, color='#ffd740', symbol='star'),
    text=[f'Max Sharpe ({ms_s:.2f})'], textposition='top right',
    name='Max Sharpe'
))
fig_ef.add_trace(go.Scatter(
    x=[mv_v * 100], y=[mv_r * 100],
    mode='markers+text',
    marker=dict(size=18, color='#00e676', symbol='diamond'),
    text=['Min Variance'], textposition='top right',
    name='Min Variance'
))

# 4. Individual stocks
eye = np.eye(n_assets)
for i, t in enumerate(tickers):
    r_s, v_s, _ = portfolio_performance(eye[i])
    fig_ef.add_trace(go.Scatter(
        x=[v_s * 100], y=[r_s * 100],
        mode='markers+text',
        marker=dict(size=9, color='#82b1ff'),
        text=[t], textposition='top center',
        showlegend=(i == 0),
        name='Individual Stocks', legendgroup='stocks'
    ))

fig_ef.update_layout(
    template='plotly_dark', height=600,
    title='📊 Efficient Frontier — Markowitz Mean-Variance Optimization',
    xaxis_title='Annualized Volatility (%)',
    yaxis_title='Annualized Return (%)'
)
fig_ef.show()

In [ ]:
# ── Optimal Weights Side-by-Side ─────────────────────────────────────────────
fig_w = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Max Sharpe Weights  (Sharpe={ms_s:.2f})',
        f'Min Variance Weights  (Vol={mv_v*100:.1f}%)'
    ]
)
fig_w.add_trace(go.Bar(
    x=tickers, y=w_ms * 100,
    marker_color=COLORS[:n_assets], name='Max Sharpe'
), row=1, col=1)
fig_w.add_trace(go.Bar(
    x=tickers, y=w_mv * 100,
    marker_color=COLORS[:n_assets], name='Min Variance'
), row=1, col=2)
fig_w.update_yaxes(title_text='Allocation (%)')
fig_w.update_layout(
    template='plotly_dark', height=380,
    title='Optimal Portfolio Weights', showlegend=False
)
fig_w.show()

In [ ]:
# ── Save Results ─────────────────────────────────────────────────────────────
ef_df = pd.DataFrame({'Volatility_pct': ef_vols, 'Return_pct': ef_rets})
ef_df.to_csv('../data/efficient_frontier.csv', index=False)

opt_df = pd.DataFrame({
    'Ticker':          tickers,
    'MaxSharpe_Weight': w_ms,
    'MinVar_Weight':    w_mv
})
opt_df.to_csv('../data/optimal_portfolio.csv', index=False)

print('💾 Saved: efficient_frontier.csv  |  optimal_portfolio.csv')
print('\n' + '='*55)
print('✅ Notebook 5 complete! Optimization Techniques Summary:')
print(f'  Critical Points  → min-variance weight w* = {w_star_clipped:.3f}')
print(f'  Lagrange         → analytical n-asset solution computed')
print(f'  scipy Max Sharpe → Sharpe = {ms_s:.3f}  (Return {ms_r*100:.1f}%, Vol {ms_v*100:.1f}%)')
print(f'  scipy Min Var    → Vol    = {mv_v*100:.2f}%  (Return {mv_r*100:.1f}%, Sharpe {mv_s:.2f})')
print('\nProceed to Notebook 06 → GBM Price Simulation (Euler Method).')